In [ ]:
# 2. Setup: imports, configuration, reproducibility
import io, re, sys, copy, time, random, pickle, shutil, hashlib, zipfile, warnings, subprocess, urllib.request
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.utils.flop_counter import FlopCounterMode
from torchvision import models, transforms as T

from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support, f1_score, roc_auc_score,
                             average_precision_score, confusion_matrix, classification_report,
                             roc_curve, precision_recall_curve)
from sklearn.preprocessing import label_binarize
from scipy.stats import binomtest

try:
    from IPython.display import display, Markdown
except ImportError:              # plain Python fallback
    display, Markdown = print, str

warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=DeprecationWarning)
sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams.update({'figure.dpi': 110, 'axes.titleweight': 'bold'})

CFG = dict(
    seed=42,
    img_size=224,               # input resolution of the ImageNet backbones
    batch_size=32,
    num_workers=2,
    val_frac=0.15, test_frac=0.15,
    near_dup_corr=0.98,         # thumbnail correlation at or above which two photos count as the same shot
    pretrained=True,            # ImageNet weights for the transfer-learning models
    scratch_epochs=30, scratch_patience=8,
    warmup_epochs=2,            # head-only epochs, backbone frozen
    finetune_epochs=15,         # full fine-tuning epochs
    patience=5,                 # early-stopping patience on validation macro-F1
    lr_head=1e-3, lr_backbone=1e-4, weight_decay=1e-2, label_smoothing=0.1, dropout=0.3,
    gflops_budget=2.0,          # compute budget per image for an edge device (Section 11)
    latency_runs=30, bootstrap=1000,
    subsample_per_class=None,   # e.g. 30 for a quick dry run of the whole notebook
)

DATA_DIR, OUT_DIR = Path('data'), Path('results')
FIG_DIR, MODEL_DIR = OUT_DIR / 'figures', OUT_DIR / 'models'
for d in (DATA_DIR, FIG_DIR, MODEL_DIR):
    d.mkdir(parents=True, exist_ok=True)

def seed_everything(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

def slug(name):
    return re.sub(r'[^a-z0-9]+', '_', name.lower()).strip('_')

def savefig(name):
    plt.tight_layout()
    plt.savefig(FIG_DIR / f'{name}.png', dpi=200, bbox_inches='tight')
    plt.show()

def pip_install(*pkgs):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *pkgs], check=False)

seed_everything(CFG['seed'])
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USE_AMP = DEVICE.type == 'cuda'
torch.backends.cudnn.benchmark = True   # faster convolutions; results can vary by about 1 point between runs

print(f'PyTorch {torch.__version__} | device: {DEVICE}' + (f' ({torch.cuda.get_device_name(0)})' if USE_AMP else ''))
if DEVICE.type == 'cpu':
    print('No GPU found. In Colab: Runtime > Change runtime type > T4 GPU, then run again.')

In [ ]:
# 3. Download TrashNet
URLS = ['https://github.com/garythung/trashnet/raw/master/data/dataset-resized.zip',
        'https://huggingface.co/datasets/garythung/trashnet/resolve/main/dataset-resized.zip']
zip_path, raw_dir = DATA_DIR / 'dataset-resized.zip', DATA_DIR / 'trashnet'

def download(urls, dest):
    for url in urls:
        try:
            print('Downloading', url)
            urllib.request.urlretrieve(url, dest)
            with zipfile.ZipFile(dest) as z:
                if z.testzip() is None:
                    return
        except Exception as e:
            print('  failed:', e)
        dest.unlink(missing_ok=True)
    raise RuntimeError('Download failed. Upload dataset-resized.zip into the data/ folder and rerun this cell.')

if not zip_path.exists():
    download(URLS, zip_path)
if not raw_dir.exists():
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(raw_dir)
print(f'{zip_path.stat().st_size / 1e6:.1f} MB archive extracted to {raw_dir}/')

In [ ]:
# 4. Index every image and audit data quality
CLASSES = ['cardboard', 'glass', 'metal', 'paper', 'plastic', 'trash']
K, TRASH = len(CLASSES), CLASSES.index('trash')

rows, skipped, corrupt = [], 0, []
for p in sorted(raw_dir.rglob('*')):
    if not p.is_file():
        continue
    if ('__MACOSX' in p.parts or p.name.startswith('.') or p.parent.name not in CLASSES
            or p.suffix.lower() not in {'.jpg', '.jpeg', '.png'}):
        skipped += 1
        continue
    try:
        with Image.open(p) as im:
            im.verify()                      # detects truncated or corrupt files
        with Image.open(p) as im:
            w, h = im.size
    except Exception:
        corrupt.append(p.name)
        continue
    rows.append(dict(path=str(p), file=p.name, label=p.parent.name, width=w, height=h,
                     md5=hashlib.md5(p.read_bytes()).hexdigest()))
df_all = pd.DataFrame(rows)
print(f'Images indexed: {len(df_all)} | system or non-image files skipped: {skipped} | corrupt: {len(corrupt)}')

# Check 2: byte-identical files stored under different labels
labels_per_file = df_all.groupby('md5')['label'].nunique()
conflict = df_all[df_all.md5.isin(labels_per_file[labels_per_file > 1].index)].sort_values('md5')
print(f'Identical files stored under two labels: {conflict.md5.nunique()} (all {len(conflict)} copies removed)')
if len(conflict):
    display(conflict[['file', 'label', 'md5']].reset_index(drop=True))
df = df_all[~df_all.md5.isin(conflict.md5)].drop_duplicates('md5').reset_index(drop=True)
print(f'Same-label duplicate copies removed: {len(df_all) - len(conflict) - len(df)}')

# Check 3: near-identical shots, grouped with union-find so they share a split
def thumb(path, size=(32, 24)):
    with Image.open(path) as im:
        a = np.asarray(im.convert('RGB').resize(size, Image.BILINEAR), dtype=np.float32).ravel()
    a -= a.mean()
    return a / (np.linalg.norm(a) + 1e-6)

Z = np.stack([thumb(p) for p in df.path])
corr = Z @ Z.T
labels_arr = np.asarray(df.label.tolist())
same_class = labels_arr[:, None] == labels_arr[None, :]
ii, jj = np.where(np.triu((corr >= CFG['near_dup_corr']) & same_class, 1))
parent = list(range(len(df)))
def find(a):
    while parent[a] != a:
        parent[a] = parent[parent[a]]
        a = parent[a]
    return a
for a, b in zip(ii, jj):
    parent[find(a)] = find(b)
df['group'] = [find(i) for i in range(len(df))]
print(f'Near-identical pairs (thumbnail correlation >= {CFG["near_dup_corr"]}): {len(ii)}, '
      f'forming {int((df.group.value_counts() > 1).sum())} groups that stay in one split')

if len(conflict):
    dup_groups = list(conflict.groupby('md5'))
    fig, axes = plt.subplots(1, len(dup_groups), figsize=(3.4 * len(dup_groups), 3))
    for a, (_, g) in zip(np.atleast_1d(axes), dup_groups):
        with Image.open(g.path.iloc[0]) as im:
            a.imshow(im.convert('RGB'))
        a.set_title('\n'.join(f'{r.file} ({r.label})' for r in g.itertuples()), fontsize=8)
        a.axis('off')
    savefig('00a_conflicting_duplicates')

if len(ii):
    top = np.argsort(-corr[ii, jj])[:4]
    fig, axes = plt.subplots(2, len(top), figsize=(3.2 * len(top), 5), squeeze=False)
    for c, t in enumerate(top):
        for r, idx in enumerate((ii[t], jj[t])):
            with Image.open(df.path[idx]) as im:
                axes[r, c].imshow(im.convert('RGB'))
            axes[r, c].set_title(df.file[idx] + (f'   r = {corr[ii[t], jj[t]]:.3f}' if r == 0 else ''), fontsize=8)
            axes[r, c].axis('off')
    savefig('00b_near_identical_shots')

if CFG['subsample_per_class']:
    df = df.groupby('label').sample(n=CFG['subsample_per_class'], random_state=CFG['seed']).reset_index(drop=True)
print(f'Clean dataset: {len(df)} images | image sizes: {df.groupby(["width", "height"]).size().to_dict()}')

In [ ]:
# 5. Exploratory analysis
counts = df.label.value_counts().reindex(CLASSES)
fig, ax = plt.subplots(figsize=(7, 3.4))
sns.barplot(x=counts.index, y=counts.values, color='#4C72B0', ax=ax)
for i, v in enumerate(counts.values):
    ax.text(i, v + 4, str(v), ha='center', fontsize=9)
ax.set(title='Images per class after cleaning', xlabel='', ylabel='images')
savefig('01_class_distribution')
print(f'Imbalance ratio, largest to smallest class: {counts.max() / counts.min():.1f} : 1')

fig, axes = plt.subplots(K, 6, figsize=(12, 1.75 * K))
for r, c in enumerate(CLASSES):
    for k, p in enumerate(df[df.label == c].sample(6, random_state=CFG['seed']).path):
        with Image.open(p) as im:
            axes[r, k].imshow(im.convert('RGB'))
        axes[r, k].axis('off')
    axes[r, 0].set_title(c, loc='left', fontsize=10)
savefig('02_samples_per_class')

In [ ]:
# 6. Group-aware stratified split: 70% train, 15% validation, 15% test
group_label = df.groupby('group').label.first()          # one entry per group; most groups are single images
g_labels = np.asarray(group_label.tolist())
g_train, g_rest = train_test_split(np.asarray(group_label.index), test_size=CFG['val_frac'] + CFG['test_frac'],
                                   stratify=g_labels, random_state=CFG['seed'])
g_val, g_test = train_test_split(g_rest, test_size=CFG['test_frac'] / (CFG['val_frac'] + CFG['test_frac']),
                                 stratify=np.asarray(group_label.loc[g_rest].tolist()), random_state=CFG['seed'])
split_of = {**{g: 'train' for g in g_train}, **{g: 'val' for g in g_val}, **{g: 'test' for g in g_test}}
df['split'] = df.group.map(split_of)
df['y'] = df.label.map(CLASSES.index)
df.to_csv(OUT_DIR / 'split.csv', index=False)

train_df, val_df, test_df = (df[df.split == s].reset_index(drop=True) for s in ('train', 'val', 'test'))
assert not (set(train_df.group) & set(test_df.group)) and not (set(train_df.group) & set(val_df.group))

split_table = pd.crosstab(df.label, df.split).reindex(index=CLASSES, columns=['train', 'val', 'test'])
split_table.loc['total'] = split_table.sum()
display(split_table)

In [ ]:
# 7. Preprocessing, augmentation and class weights
S = CFG['img_size']
MEAN, STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)      # ImageNet statistics

train_tf = T.Compose([
    T.RandomResizedCrop(S, scale=(0.7, 1.0), ratio=(0.75, 1.33)),
    T.RandomHorizontalFlip(),
    T.RandomVerticalFlip(),
    T.RandomRotation(15, fill=255),
    T.ColorJitter(brightness=0.25, contrast=0.25, saturation=0.2, hue=0.02),
    T.ToTensor(),
    T.Normalize(MEAN, STD),
])
eval_tf = T.Compose([T.Resize((S, S)), T.ToTensor(), T.Normalize(MEAN, STD)])

def load_rgb(path):
    with Image.open(path) as im:
        return im.convert('RGB')

class WasteDataset(Dataset):
    def __init__(self, frame, tf):
        self.paths, self.ys, self.tf = frame.path.tolist(), frame.y.tolist(), tf
    def __len__(self):
        return len(self.paths)
    def __getitem__(self, i):
        return self.tf(load_rgb(self.paths[i])), self.ys[i]

def make_loader(frame, tf, train=False):
    return DataLoader(WasteDataset(frame, tf), batch_size=CFG['batch_size'], shuffle=train, drop_last=train,
                      num_workers=CFG['num_workers'], pin_memory=USE_AMP,
                      persistent_workers=CFG['num_workers'] > 0)

train_loader = make_loader(train_df, train_tf, train=True)
val_loader, test_loader = make_loader(val_df, eval_tf), make_loader(test_df, eval_tf)

# Inverse-frequency class weights with mean 1: a mistake on a trash image costs more than one on paper
n_train = np.bincount(train_df.y.to_numpy(), minlength=K)
class_weights = torch.tensor(n_train.sum() / (K * n_train), dtype=torch.float32)
display(pd.DataFrame({'train images': n_train, 'loss weight': class_weights.numpy().round(2)}, index=CLASSES))

def denorm(t):
    return (t.permute(1, 2, 0).numpy() * STD + MEAN).clip(0, 1)

base = load_rgb(train_df.path.iloc[0])
fig, axes = plt.subplots(1, 6, figsize=(12, 2.3))
axes[0].imshow(base.resize((S, S))); axes[0].set_title('original', fontsize=9)
for a in axes[1:]:
    a.imshow(denorm(train_tf(base))); a.set_title('augmented', fontsize=9)
for a in axes:
    a.axis('off')
savefig('03_augmentation')

In [ ]:
# 8a. Metrics and cost measurements
def classification_metrics(y_true, y_prob):
    y_pred = y_prob.argmax(1)
    p, r, f, _ = precision_recall_fscore_support(y_true, y_pred, average='macro', zero_division=0)
    return {'Accuracy': accuracy_score(y_true, y_pred),
            'Precision (macro)': p, 'Recall (macro)': r, 'F1 (macro)': f,
            'F1 (weighted)': f1_score(y_true, y_pred, average='weighted', zero_division=0),
            'ROC-AUC (macro OvR)': roc_auc_score(y_true, y_prob, multi_class='ovr', average='macro', labels=list(range(K))),
            'mAP (macro AP)': average_precision_score(label_binarize(y_true, classes=list(range(K))), y_prob, average='macro')}

def sorting_metrics(y_true, y_pred):
    to_recycling, is_recyclable = y_pred != TRASH, y_true != TRASH
    return {'Contamination': ((y_true == TRASH) & to_recycling).sum() / max(to_recycling.sum(), 1),
            'Recyclable loss': (is_recyclable & ~to_recycling).sum() / max(is_recyclable.sum(), 1),
            'Cross-material error': (is_recyclable & to_recycling & (y_true != y_pred)).sum()
                                    / max((is_recyclable & to_recycling).sum(), 1)}

def count_params(model):
    return sum(p.numel() for p in model.parameters()) / 1e6

def size_mb(model):
    buf = io.BytesIO()
    torch.save(model.state_dict(), buf)
    return buf.getbuffer().nbytes / 1e6

def gflops(model):
    m = copy.deepcopy(model).float().cpu().eval()
    with FlopCounterMode(display=False) as fc, torch.no_grad():
        m(torch.randn(1, 3, S, S))
    return fc.get_total_flops() / 1e9

SAMPLE_IMG = load_rgb(test_df.path.iloc[0])

def latency_ms(predict_fn, warmup=3):
    """Median time for one image, decoded photo to probabilities, on one CPU thread."""
    threads = torch.get_num_threads()
    torch.set_num_threads(1)
    try:
        for _ in range(warmup):
            predict_fn(SAMPLE_IMG)
        times = []
        for _ in range(CFG['latency_runs']):
            t0 = time.perf_counter()
            predict_fn(SAMPLE_IMG)
            times.append((time.perf_counter() - t0) * 1000)
    finally:
        torch.set_num_threads(threads)
    return float(np.median(times))

def torch_cpu_predictor(model):
    m = copy.deepcopy(model).float().cpu().eval()
    def predict(img):
        with torch.inference_mode():
            return m(eval_tf(img).unsqueeze(0)).softmax(1)
    return predict

RESULTS = {}
def record(name, role, y_true, y_prob, params, size, flops, latency, minutes, val_f1=np.nan):
    RESULTS[name] = dict(role=role, y_true=y_true, y_prob=y_prob,
                         **classification_metrics(y_true, y_prob), **sorting_metrics(y_true, y_prob.argmax(1)),
                         **{'Params (M)': params, 'Size (MB)': size, 'GFLOPs': flops, 'CPU latency (ms)': latency,
                            'Train time (min)': minutes, 'Val F1 (macro)': val_f1})
    r = RESULTS[name]
    print(f'{name}: test accuracy {r["Accuracy"]:.4f} | macro-F1 {r["F1 (macro)"]:.4f} | '
          f'ROC-AUC {r["ROC-AUC (macro OvR)"]:.4f} | CPU latency {latency:.1f} ms')

In [ ]:
# 8b. Training loop shared by every neural model
def autocast():
    return torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP)

def train_one_epoch(model, criterion, optimizer, scaler):
    model.train()
    total, correct, n = 0.0, 0, 0
    for x, y in train_loader:
        x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
        with autocast():
            logits = model(x)
            loss = criterion(logits.float(), y)
        optimizer.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        total += loss.item() * len(y); correct += (logits.argmax(1) == y).sum().item(); n += len(y)
    return total / n, correct / n

@torch.inference_mode()
def validate(model, criterion):
    model.eval()
    total, n, preds, ys = 0.0, 0, [], []
    for x, y in val_loader:
        x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
        with autocast():
            logits = model(x)
        total += criterion(logits.float(), y).item() * len(y); n += len(y)
        preds.append(logits.argmax(1).cpu()); ys.append(y.cpu())
    y_true, y_pred = torch.cat(ys).numpy(), torch.cat(preds).numpy()
    return total / n, accuracy_score(y_true, y_pred), f1_score(y_true, y_pred, average='macro', zero_division=0)

@torch.inference_mode()
def predict_proba(model, loader):
    model.eval()
    probs, ys = [], []
    for x, y in loader:
        with autocast():
            probs.append(model(x.to(DEVICE, non_blocking=True)).float().softmax(1).cpu())
        ys.append(y)
    return torch.cat(ys).numpy(), torch.cat(probs).numpy()

def fit(model, name, stages):
    """Runs a list of training stages. Each stage sets its epochs, patience, AdamW parameter groups and which
    parameters stay trainable (None = all). Early stopping watches validation macro-F1; the best checkpoint
    is restored at the end and saved to disk."""
    model = model.to(DEVICE)
    criterion = nn.CrossEntropyLoss(weight=class_weights.to(DEVICE), label_smoothing=CFG['label_smoothing'])
    scaler = torch.amp.GradScaler(DEVICE.type, enabled=USE_AMP)
    history, best, t0 = [], {'f1': -1.0, 'epoch': 0, 'state': None}, time.time()
    for st in stages:
        keep = None if st['trainable'] is None else {id(p) for p in st['trainable']}
        for p in model.parameters():
            p.requires_grad_(keep is None or id(p) in keep)
        optimizer = torch.optim.AdamW(st['groups'], weight_decay=CFG['weight_decay'])
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(st['epochs'], 1))
        stale = 0
        for _ in range(st['epochs']):
            tr_loss, tr_acc = train_one_epoch(model, criterion, optimizer, scaler)
            va_loss, va_acc, va_f1 = validate(model, criterion)
            scheduler.step()
            ep = len(history) + 1
            history.append(dict(epoch=ep, stage=st['name'], train_loss=tr_loss, train_acc=tr_acc,
                                val_loss=va_loss, val_acc=va_acc, val_f1=va_f1))
            mark = ''
            if va_f1 > best['f1']:
                best.update(f1=va_f1, epoch=ep, state=copy.deepcopy(model.state_dict()))
                stale, mark = 0, '  <- best'
            else:
                stale += 1
            print(f'{name:<18} {st["name"]:<8} epoch {ep:2d} | train loss {tr_loss:.3f} acc {tr_acc:.3f} | '
                  f'val loss {va_loss:.3f} acc {va_acc:.3f} macro-F1 {va_f1:.3f}{mark}')
            if stale >= st['patience']:
                print(f'{name}: early stop, no validation gain for {stale} epochs')
                break
    for p in model.parameters():
        p.requires_grad_(True)
    model.load_state_dict(best['state'])
    minutes = (time.time() - t0) / 60
    torch.save(model.state_dict(), MODEL_DIR / f'{slug(name)}.pt')
    print(f'{name}: best validation macro-F1 {best["f1"]:.4f} at epoch {best["epoch"]} | {minutes:.1f} min')
    return model, pd.DataFrame(history), minutes, best['f1']

def plot_history(hist, name):
    fig, axes = plt.subplots(1, 2, figsize=(11, 3.4))
    axes[0].plot(hist.epoch, hist.train_loss, label='train')
    axes[0].plot(hist.epoch, hist.val_loss, label='validation')
    axes[0].set(title=f'{name}: loss', xlabel='epoch')
    axes[1].plot(hist.epoch, hist.train_acc, label='train accuracy')
    axes[1].plot(hist.epoch, hist.val_acc, label='validation accuracy')
    axes[1].plot(hist.epoch, hist.val_f1, '--', label='validation macro-F1')
    axes[1].set(title=f'{name}: accuracy and F1', xlabel='epoch', ylim=(0, 1.02))
    if (hist.stage == 'warmup').any():
        for a in axes:
            a.axvline(hist.loc[hist.stage == 'warmup', 'epoch'].max() + 0.5, color='grey', ls=':', lw=1)
    for a in axes:
        a.legend(fontsize=8)
    savefig(f'curves_{slug(name)}')

In [ ]:
# 9. Baseline 1: HOG + colour + LBP features with an RBF SVM
from skimage.feature import hog, local_binary_pattern
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.svm import SVC

def handcrafted(img):
    img = img.resize((128, 128), Image.BILINEAR)
    gray = np.asarray(img.convert('L'))
    f_shape = hog(gray, orientations=9, pixels_per_cell=(16, 16), cells_per_block=(2, 2), block_norm='L2-Hys')
    hsv = np.asarray(img.convert('HSV')).reshape(-1, 3)
    f_colour, _ = np.histogramdd(hsv, bins=(16, 4, 4), range=((0, 256),) * 3)
    lbp = local_binary_pattern(gray, P=8, R=1, method='uniform')
    f_texture, _ = np.histogram(lbp, bins=np.arange(11), density=True)
    return np.concatenate([f_shape, f_colour.ravel() / f_colour.sum(), f_texture]).astype(np.float32)

t0 = time.time()
X_tr, X_va, X_te = (np.stack([handcrafted(load_rgb(p)) for p in d.path]) for d in (train_df, val_df, test_df))
print(f'{X_tr.shape[1]} features per image | extraction {time.time() - t0:.0f} s')

def svm_pipeline(C, prob=False):
    return make_pipeline(StandardScaler(), PCA(n_components=0.95, random_state=CFG['seed']),
                         SVC(C=C, kernel='rbf', gamma='scale', class_weight='balanced',
                             probability=prob, random_state=CFG['seed']))

val_f1_by_C = {}
for C in (1, 10, 100):
    val_f1_by_C[C] = f1_score(val_df.y, svm_pipeline(C).fit(X_tr, train_df.y).predict(X_va), average='macro')
    print(f'C = {C:<3}  validation macro-F1 {val_f1_by_C[C]:.4f}')
best_C = max(val_f1_by_C, key=val_f1_by_C.get)

t0 = time.time()
svm = svm_pipeline(best_C, prob=True).fit(X_tr, train_df.y)
svm_minutes = (time.time() - t0) / 60
with open(MODEL_DIR / 'hog_svm.pkl', 'wb') as fh:
    pickle.dump(svm, fh)
record('HOG + SVM', 'baseline', test_df.y.values, svm.predict_proba(X_te),
       params=np.nan, size=(MODEL_DIR / 'hog_svm.pkl').stat().st_size / 1e6, flops=np.nan,
       latency=latency_ms(lambda img: svm.predict_proba(handcrafted(img)[None])),
       minutes=svm_minutes, val_f1=val_f1_by_C[best_C])

In [ ]:
# 10. Baseline 2: CNN trained from scratch
class ScratchCNN(nn.Module):
    def __init__(self, widths=(32, 64, 128, 256, 256)):
        super().__init__()
        layers, c_in = [], 3
        for c_out in widths:
            layers += [nn.Conv2d(c_in, c_out, 3, padding=1, bias=False), nn.BatchNorm2d(c_out),
                       nn.ReLU(inplace=True), nn.MaxPool2d(2)]
            c_in = c_out
        self.features = nn.Sequential(*layers)
        self.head = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Dropout(CFG['dropout']), nn.Linear(c_in, K))
    def forward(self, x):
        return self.head(self.features(x))

seed_everything(CFG['seed'])
scratch = ScratchCNN()
scratch, hist_scratch, t_scratch, vf1_scratch = fit(scratch, 'Scratch CNN', [dict(
    name='scratch', epochs=CFG['scratch_epochs'], patience=CFG['scratch_patience'],
    groups=[{'params': list(scratch.parameters()), 'lr': 1e-3}], trainable=None)])
plot_history(hist_scratch, 'Scratch CNN')
record('Scratch CNN', 'baseline', *predict_proba(scratch, test_loader), params=count_params(scratch),
       size=size_mb(scratch), flops=gflops(scratch), latency=latency_ms(torch_cpu_predictor(scratch)),
       minutes=t_scratch, val_f1=vf1_scratch)

In [ ]:
# 11. Model builder and the two-stage transfer-learning recipe
def build_model(arch):
    """ImageNet-pretrained backbone with a new 6-class head. Returns (model, head)."""
    pre = CFG['pretrained']
    if arch == 'ResNet-50':
        m = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2 if pre else None)
        m.fc = nn.Sequential(nn.Dropout(CFG['dropout']), nn.Linear(m.fc.in_features, K))
        head = m.fc
    elif arch == 'MobileNetV3-Large':
        m = models.mobilenet_v3_large(weights=models.MobileNet_V3_Large_Weights.IMAGENET1K_V2 if pre else None)
        m.classifier[2] = nn.Dropout(CFG['dropout'])
        m.classifier[3] = nn.Linear(m.classifier[3].in_features, K)
        head = m.classifier
    elif arch == 'EfficientNet-B0':
        m = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1 if pre else None)
        m.classifier = nn.Sequential(nn.Dropout(CFG['dropout']), nn.Linear(m.classifier[1].in_features, K))
        head = m.classifier
    else:
        raise ValueError(arch)
    return m, head

def gradcam_layer(model, arch):
    return model.layer4[-1] if arch == 'ResNet-50' else model.features[-1]

def train_transfer(arch):
    seed_everything(CFG['seed'])
    model, head = build_model(arch)
    head_params = list(head.parameters())
    head_ids = {id(p) for p in head_params}
    backbone_params = [p for p in model.parameters() if id(p) not in head_ids]
    stages = [dict(name='warmup', epochs=CFG['warmup_epochs'], patience=CFG['patience'],
                   groups=[{'params': head_params, 'lr': CFG['lr_head']}], trainable=head_params),
              dict(name='finetune', epochs=CFG['finetune_epochs'], patience=CFG['patience'],
                   groups=[{'params': backbone_params, 'lr': CFG['lr_backbone']},
                           {'params': head_params, 'lr': CFG['lr_head']}], trainable=None)]
    model, hist, minutes, vf1 = fit(model, arch, stages)
    plot_history(hist, arch)
    record(arch, 'candidate', *predict_proba(model, test_loader), params=count_params(model), size=size_mb(model),
           flops=gflops(model), latency=latency_ms(torch_cpu_predictor(model)), minutes=minutes, val_f1=vf1)
    return model

trained = {}

In [ ]:
trained['ResNet-50'] = train_transfer('ResNet-50')

In [ ]:
trained['MobileNetV3-Large'] = train_transfer('MobileNetV3-Large')

In [ ]:
trained['EfficientNet-B0'] = train_transfer('EfficientNet-B0')

In [ ]:
# Apply the selection rule from Section 11 (validation data only)
cand = pd.DataFrame({a: {'Val F1 (macro)': RESULTS[a]['Val F1 (macro)'], 'GFLOPs': RESULTS[a]['GFLOPs'],
                         'CPU latency (ms)': RESULTS[a]['CPU latency (ms)']} for a in trained}).T
cand['Within budget'] = cand['GFLOPs'] <= CFG['gflops_budget']
pool = cand[cand['Within budget']] if cand['Within budget'].any() else cand
SELECTED = pool.sort_values(['Val F1 (macro)', 'GFLOPs'], ascending=[False, True]).index[0]
cand['Selected'] = cand.index == SELECTED
display(cand)
print(f'Selected for deployment: {SELECTED}')
model_sel = trained[SELECTED]

In [ ]:
# 13. Test-set comparison of all five models
PERF = ['Accuracy', 'Precision (macro)', 'Recall (macro)', 'F1 (macro)', 'F1 (weighted)', 'ROC-AUC (macro OvR)', 'mAP (macro AP)']
SORT = ['Contamination', 'Recyclable loss', 'Cross-material error']
COST = ['Params (M)', 'Size (MB)', 'GFLOPs', 'CPU latency (ms)', 'Train time (min)']
table = pd.DataFrame({n: {c: r[c] for c in PERF + SORT + COST} for n, r in RESULTS.items()}).T.astype(float)
table.insert(0, 'Role', ['selected' if n == SELECTED else r['role'] for n, r in RESULTS.items()])
table.to_csv(OUT_DIR / 'model_comparison.csv')
display(table.style.format('{:.4f}', subset=PERF + SORT).format('{:.2f}', subset=COST, na_rep='-')
        .highlight_max(subset=PERF, color='#cfe8d5').highlight_min(subset=SORT, color='#cfe8d5'))

fig, axes = plt.subplots(1, 2, figsize=(13, 4.3))
table[['Accuracy', 'F1 (macro)', 'ROC-AUC (macro OvR)', 'mAP (macro AP)']].plot.bar(ax=axes[0], rot=12, width=0.8)
axes[0].set(title='Test-set performance', ylim=(0, 1.05))
axes[0].legend(fontsize=8, loc='lower right')
for n, r in table.iterrows():
    area = 30 + 8 * (r['Params (M)'] if pd.notna(r['Params (M)']) else 1)
    axes[1].scatter(r['CPU latency (ms)'], r['Accuracy'], s=area, alpha=0.75, edgecolor='k')
    axes[1].annotate(n, (r['CPU latency (ms)'], r['Accuracy']), xytext=(6, 4), textcoords='offset points', fontsize=8)
axes[1].set(xscale='log', xlabel='CPU latency per image, one thread (ms, log scale)', ylabel='test accuracy',
            title='Accuracy against speed (marker area follows parameters)')
savefig('10_model_comparison')

In [ ]:
# 14. Bootstrap confidence intervals and McNemar's exact test against the selected model
def bootstrap_ci(y, prob, metric, seed=0):
    rng, pred = np.random.default_rng(seed), prob.argmax(1)
    vals = [metric(y[i], pred[i]) for i in (rng.integers(0, len(y), len(y)) for _ in range(CFG['bootstrap']))]
    return np.percentile(vals, [2.5, 97.5])

def mcnemar_exact(y, pred_a, pred_b):
    a_only = int(((pred_a == y) & (pred_b != y)).sum())
    b_only = int(((pred_a != y) & (pred_b == y)).sum())
    p = binomtest(min(a_only, b_only), a_only + b_only, 0.5).pvalue if a_only + b_only else 1.0
    return a_only, b_only, p

macro_f1 = lambda a, b: f1_score(a, b, average='macro', zero_division=0)
y_test = RESULTS[SELECTED]['y_true']
pred_sel = RESULTS[SELECTED]['y_prob'].argmax(1)
CI, MCN, rows = {}, {}, []
for n, r in RESULTS.items():
    assert np.array_equal(r['y_true'], y_test)
    CI[n] = bootstrap_ci(y_test, r['y_prob'], accuracy_score)
    f_lo, f_hi = bootstrap_ci(y_test, r['y_prob'], macro_f1)
    row = {'Model': n, 'Accuracy': round(r['Accuracy'], 4), 'Accuracy 95% CI': f'{CI[n][0]:.3f} to {CI[n][1]:.3f}',
           'Macro-F1': round(r['F1 (macro)'], 4), 'Macro-F1 95% CI': f'{f_lo:.3f} to {f_hi:.3f}'}
    if n != SELECTED:
        a, b, p = mcnemar_exact(y_test, pred_sel, r['y_prob'].argmax(1))
        MCN[n] = p
        row.update({'Only selected right': a, 'Only this model right': b, 'McNemar p': round(p, 4),
                    'Significant at 5%': 'yes' if p < 0.05 else 'no'})
    rows.append(row)
sig_table = pd.DataFrame(rows).set_index('Model')
sig_table.to_csv(OUT_DIR / 'significance.csv')
display(sig_table)

In [ ]:
# 15. Per-class performance of the selected model
yp_sel = RESULTS[SELECTED]['y_prob']
report = pd.DataFrame(classification_report(y_test, pred_sel, labels=list(range(K)), target_names=CLASSES,
                                            output_dict=True, zero_division=0)).T
report.to_csv(OUT_DIR / 'per_class_report.csv')
display(report.round(3))

def plot_cm(name, ax):
    cm = confusion_matrix(y_test, RESULTS[name]['y_prob'].argmax(1), labels=list(range(K)))
    share = cm / np.maximum(cm.sum(1, keepdims=True), 1)
    annot = np.array([[f'{c}\n{s:.0%}' for c, s in zip(rc, rs)] for rc, rs in zip(cm, share)])
    sns.heatmap(share, annot=annot, fmt='', cmap='Blues', vmin=0, vmax=1, cbar=False,
                xticklabels=CLASSES, yticklabels=CLASSES, ax=ax, annot_kws={'fontsize': 8})
    ax.set(title=f'{name} (count and row share)', xlabel='predicted', ylabel='true')

fig, axes = plt.subplots(1, 2, figsize=(12, 4.8))
plot_cm('HOG + SVM', axes[0])
plot_cm(SELECTED, axes[1])
savefig('11_confusion_matrices')

y_bin = label_binarize(y_test, classes=list(range(K)))
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for k, c in enumerate(CLASSES):
    fpr, tpr, _ = roc_curve(y_bin[:, k], yp_sel[:, k])
    axes[0].plot(fpr, tpr, label=f'{c} (AUC {roc_auc_score(y_bin[:, k], yp_sel[:, k]):.3f})')
    prec, rec, _ = precision_recall_curve(y_bin[:, k], yp_sel[:, k])
    axes[1].plot(rec, prec, label=f'{c} (AP {average_precision_score(y_bin[:, k], yp_sel[:, k]):.3f})')
axes[0].plot([0, 1], [0, 1], 'k:', lw=1)
axes[0].set(title=f'{SELECTED}: ROC, one-vs-rest', xlabel='false positive rate', ylabel='true positive rate')
axes[1].set(title=f'{SELECTED}: precision-recall', xlabel='recall', ylabel='precision')
for a in axes:
    a.legend(fontsize=8)
savefig('12_roc_pr_curves')

cm_off = confusion_matrix(y_test, pred_sel, labels=list(range(K)))
np.fill_diagonal(cm_off, 0)
top_confusions = sorted(((int(cm_off[i, j]), CLASSES[i], CLASSES[j]) for i in range(K) for j in range(K) if cm_off[i, j]),
                        reverse=True)[:5]
print('Most frequent confusions (true -> predicted):')
for n_img, t_cls, p_cls in top_confusions:
    print(f'  {t_cls:>9} -> {p_cls:<9} {n_img} images')
print('Sorting impact:', {k: f'{RESULTS[SELECTED][k]:.2%}' for k in SORT})

In [ ]:
# 16. Grad-CAM on the selected model
class GradCAM:
    def __init__(self, model, layer):
        self.model, self.acts, self.grads = model, None, None
        self.handle = layer.register_forward_hook(self._hook)
    def _hook(self, module, inputs, output):
        self.acts = output
        if output.requires_grad:
            output.register_hook(lambda g: setattr(self, 'grads', g))
    def __call__(self, x):
        self.model.eval()
        with torch.enable_grad():
            logits = self.model(x)
            cls = int(logits.argmax(1))
            self.model.zero_grad(set_to_none=True)
            logits[0, cls].backward()
        weights = self.grads.mean(dim=(2, 3), keepdim=True)
        cam = F.relu((weights * self.acts).sum(1, keepdim=True))
        cam = F.interpolate(cam, size=x.shape[-2:], mode='bilinear', align_corners=False)[0, 0]
        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        return cam.detach().cpu().numpy(), cls, float(logits.softmax(1)[0, cls])
    def remove(self):
        self.handle.remove()

cam_model = copy.deepcopy(model_sel).float().to(DEVICE).eval()
gradcam = GradCAM(cam_model, gradcam_layer(cam_model, SELECTED))
conf_sel = yp_sel.max(1)
right = []
for k in range(K):
    ok = np.flatnonzero((y_test == k) & (pred_sel == k))
    if len(ok):
        right.append(ok[np.argmax(conf_sel[ok])])
wrong = list(np.flatnonzero(pred_sel != y_test)[:6])

fig, axes = plt.subplots(2, 6, figsize=(15, 5.8))
for a in axes.ravel():
    a.axis('off')
for row, idxs, colour in ((0, right, 'green'), (1, wrong, 'red')):
    for a, i in zip(axes[row], idxs):
        img = load_rgb(test_df.path.iloc[i])
        cam, cls, p = gradcam(eval_tf(img).unsqueeze(0).to(DEVICE))
        a.imshow(img.resize((S, S)))
        a.imshow(cam, cmap='jet', alpha=0.4)
        a.set_title(f'true: {CLASSES[y_test[i]]}\npredicted: {CLASSES[cls]} ({p:.0%})', fontsize=8, color=colour)
gradcam.remove()
fig.suptitle(f'{SELECTED} Grad-CAM. Top: correct. Bottom: errors.', fontsize=11)
savefig('13_gradcam')
print(f'{len(wrong)} of {int((pred_sel != y_test).sum())} misclassified test images shown.')

In [ ]:
# 17. ONNX export + INT8 post-training static quantisation (ONNX Runtime)
DEPLOY = None
try:
    try:
        import onnx, onnxruntime as ort
    except ImportError:
        pip_install('onnx', 'onnxruntime')
        import onnx, onnxruntime as ort
    from onnxruntime.quantization import quantize_static, CalibrationDataReader, QuantFormat, QuantType

    fp32_path, int8_path = MODEL_DIR / 'selected_fp32.onnx', MODEL_DIR / 'selected_int8.onnx'
    m_cpu = copy.deepcopy(model_sel).float().cpu().eval()
    dummy = torch.randn(1, 3, S, S)
    try:
        torch.onnx.export(m_cpu, dummy, str(fp32_path), input_names=['input'], output_names=['logits'], opset_version=17)
    except Exception as e:            # some PyTorch builds route through the dynamo exporter, which needs onnxscript
        print('Default ONNX export failed, retrying with the dynamo exporter:', type(e).__name__)
        pip_install('onnxscript')
        torch.onnx.export(m_cpu, (dummy,), str(fp32_path), input_names=['input'], output_names=['logits'], dynamo=True)
    onnx.checker.check_model(str(fp32_path))

    def session(path):
        so = ort.SessionOptions()
        so.intra_op_num_threads, so.inter_op_num_threads = 1, 1
        return ort.InferenceSession(str(path), sess_options=so, providers=['CPUExecutionProvider'])
    IN_NAME = session(fp32_path).get_inputs()[0].name

    class CalibrationImages(CalibrationDataReader):
        def __init__(self, paths):
            self.it = iter(paths)
        def get_next(self):
            p = next(self.it, None)
            return None if p is None else {IN_NAME: eval_tf(load_rgb(p)).unsqueeze(0).numpy()}

    per_class = int(min(20, train_df.label.value_counts().min()))
    calib_paths = train_df.groupby('label').sample(n=per_class, random_state=CFG['seed']).path.tolist()
    quantize_static(str(fp32_path), str(int8_path), CalibrationImages(calib_paths), quant_format=QuantFormat.QDQ,
                    per_channel=True, activation_type=QuantType.QInt8, weight_type=QuantType.QInt8)

    def ort_predictor(sess):
        return lambda img: sess.run(None, {IN_NAME: eval_tf(img).unsqueeze(0).numpy()})[0]

    rows = [{'Format': 'PyTorch FP32', 'Size (MB)': RESULTS[SELECTED]['Size (MB)'],
             'CPU latency (ms)': RESULTS[SELECTED]['CPU latency (ms)'], 'Accuracy': RESULTS[SELECTED]['Accuracy'],
             'F1 (macro)': RESULTS[SELECTED]['F1 (macro)'], 'Agreement with PyTorch': 1.0}]
    for fmt, path in (('ONNX FP32', fp32_path), ('ONNX INT8', int8_path)):
        sess = session(path)
        predict = ort_predictor(sess)
        pred = np.concatenate([predict(load_rgb(p)) for p in test_df.path]).argmax(1)
        rows.append({'Format': fmt, 'Size (MB)': path.stat().st_size / 1e6, 'CPU latency (ms)': latency_ms(predict),
                     'Accuracy': accuracy_score(y_test, pred), 'F1 (macro)': macro_f1(y_test, pred),
                     'Agreement with PyTorch': (pred == pred_sel).mean()})
    DEPLOY = pd.DataFrame(rows).set_index('Format')
    DEPLOY.to_csv(OUT_DIR / 'deployment_optimisation.csv')
    display(DEPLOY.round(4))
except Exception as e:
    print('ONNX / INT8 step did not complete:', repr(e))
    print('The rest of the notebook does not depend on it.')

In [ ]:
# 18. Coverage against accuracy, threshold chosen on validation data
_, val_prob_sel = predict_proba(model_sel, val_loader)
val_conf, val_ok = val_prob_sel.max(1), val_prob_sel.argmax(1) == val_df.y.values
test_conf, test_ok = yp_sel.max(1), pred_sel == y_test
TARGET = 0.99
curve = pd.DataFrame([{'threshold': t,
                       'val coverage': (val_conf >= t).mean(),
                       'val accuracy': val_ok[val_conf >= t].mean() if (val_conf >= t).any() else np.nan,
                       'test coverage': (test_conf >= t).mean(),
                       'test accuracy': test_ok[test_conf >= t].mean() if (test_conf >= t).any() else np.nan}
                      for t in np.round(np.arange(0.30, 0.96, 0.05), 2)])
valid = curve.dropna(subset=['val accuracy'])
meets = valid[valid['val accuracy'] >= TARGET]
wide = valid[valid['val coverage'] >= 0.8]
if len(meets):
    THR, rule = float(meets.threshold.iloc[0]), f'lowest threshold with validation accuracy of at least {TARGET:.0%}'
elif len(wide):
    THR, rule = float(wide.loc[wide['val accuracy'].idxmax(), 'threshold']), 'most accurate threshold that still auto-sorts 80% of validation items'
elif len(valid):
    THR, rule = float(valid.loc[valid['val accuracy'].idxmax(), 'threshold']), 'most accurate threshold on the validation set'
else:
    THR, rule = float(curve.threshold.min()), 'lowest threshold on the grid; the model was never confident enough for the target'
op = curve.set_index('threshold').loc[THR]
curve.to_csv(OUT_DIR / 'reject_option.csv', index=False)

fig, ax = plt.subplots(figsize=(7, 3.8))
ax.plot(curve.threshold, curve['test accuracy'], 'o-', label='accuracy on auto-sorted items')
ax.plot(curve.threshold, curve['test coverage'], 's--', label='share of items auto-sorted')
ax.axvline(THR, color='grey', ls=':', lw=1)
ax.set(xlabel='confidence threshold', ylim=(0, 1.02), title='Reject option on the test set')
ax.legend(fontsize=8)
savefig('14_reject_option')
auto_acc = f'{op["test accuracy"]:.1%}' if pd.notna(op['test accuracy']) else 'not defined (no item passed the threshold)'
print(f'Threshold {THR:.2f} ({rule}): the model sorts {op["test coverage"]:.1%} of test items automatically '
      f'at {auto_acc} accuracy; the rest go to manual sorting.')

In [ ]:
# 19. Photo in, bin decision out
BIN = {'cardboard': 'Dry: paper and cardboard', 'paper': 'Dry: paper and cardboard', 'plastic': 'Dry: plastic',
       'glass': 'Dry: glass', 'metal': 'Dry: metal', 'trash': 'Residual (non-recyclable)'}

def classify(photo, threshold=None):
    """photo: file path or PIL image."""
    threshold = THR if threshold is None else threshold
    img = photo if isinstance(photo, Image.Image) else load_rgb(photo)
    model_sel.eval()
    with torch.inference_mode():
        p = model_sel(eval_tf(img.convert('RGB')).unsqueeze(0).to(DEVICE)).float().softmax(1)[0].cpu().numpy()
    top = p.argsort()[::-1][:3]
    best = CLASSES[int(top[0])]
    return {'class': best, 'confidence': round(float(p[top[0]]), 3),
            'bin': BIN[best] if p[top[0]] >= threshold else 'Manual check (low confidence)',
            'top3': [(CLASSES[i], round(float(p[i]), 3)) for i in top]}

for path in test_df.sample(3, random_state=1).path:
    print(Path(path).name, '->', classify(path))

RUN_DEMO = False
if RUN_DEMO:
    pip_install('gradio')
    import gradio as gr
    def demo(img):
        out = classify(img)
        return dict(out['top3']), out['bin']
    gr.Interface(fn=demo, inputs=gr.Image(type='pil', label='Photo of one item'),
                 outputs=[gr.Label(num_top_classes=3, label='Material'), gr.Textbox(label='Bin')],
                 title='Smart Waste Classification').launch(share=True)

In [ ]:
# 20. Results summary written from this run's numbers
R, sel = RESULTS, SELECTED
pct = lambda v: f'{100 * v:.1f}%'
pts = lambda d: f'{abs(100 * d):.1f} percentage points'

def compare(a, b, key, noun, lower_word, higher_word):
    va, vb = R[a][key], R[b][key]
    if max(va, vb) / max(min(va, vb), 1e-9) < 1.15:
        return f'about the same {noun}'
    return f'{vb / va:.1f}x {lower_word} {noun}' if va < vb else f'{va / vb:.1f}x {higher_word} {noun}'

def speed(a, b):
    va, vb = R[a]['CPU latency (ms)'], R[b]['CPU latency (ms)']
    if max(va, vb) / max(min(va, vb), 1e-9) < 1.15:
        return 'at about the same speed'
    return f'{vb / va:.1f}x faster' if va < vb else f'{va / vb:.1f}x slower'

def diff_phrase(a, b):
    d = R[a]['Accuracy'] - R[b]['Accuracy']
    return 'the same accuracy' if abs(d) < 1e-9 else f'{pts(d)} {"higher" if d > 0 else "lower"} accuracy'

text = []
text.append(f"**Test performance.** On the held-out test set of {len(y_test)} images, the selected model, {sel}, reached "
            f"{pct(R[sel]['Accuracy'])} accuracy (95% bootstrap CI {pct(CI[sel][0])} to {pct(CI[sel][1])}), macro precision "
            f"{R[sel]['Precision (macro)']:.3f}, macro recall {R[sel]['Recall (macro)']:.3f}, macro-F1 {R[sel]['F1 (macro)']:.3f}, "
            f"macro ROC-AUC {R[sel]['ROC-AUC (macro OvR)']:.3f} and mAP {R[sel]['mAP (macro AP)']:.3f}.")

best_base = max(['HOG + SVM', 'Scratch CNN'], key=lambda n: R[n]['Accuracy'])
text.append(f"**Baselines.** The HOG + SVM baseline reached {pct(R['HOG + SVM']['Accuracy'])} accuracy (macro-F1 "
            f"{R['HOG + SVM']['F1 (macro)']:.3f}) and the CNN trained from scratch reached {pct(R['Scratch CNN']['Accuracy'])} "
            f"(macro-F1 {R['Scratch CNN']['F1 (macro)']:.3f}). The selected transfer-learning model has {diff_phrase(sel, best_base)} "
            f"than the stronger baseline ({best_base}). "
            + (f"The hand-crafted features beat the scratch CNN: {len(train_df):,} training images are too few to learn visual "
               f"features from random initialisation, which is the case for transfer learning."
               if R['HOG + SVM']['Accuracy'] > R['Scratch CNN']['Accuracy'] else
               f"The scratch CNN matched the hand-crafted features, and both stayed below the pretrained models: {len(train_df):,} "
               f"training images are too few to learn visual features from random initialisation."
               if abs(R['HOG + SVM']['Accuracy'] - R['Scratch CNN']['Accuracy']) < 1e-9 else
               "The scratch CNN beat the hand-crafted features but stayed below the pretrained models, which is the case for transfer learning."))

for alt in [n for n in trained if n != sel]:
    p = MCN[alt]
    text.append(f"**{sel} against {alt}.** {pct(R[sel]['Accuracy'])} against {pct(R[alt]['Accuracy'])}, "
                f"{diff_phrase(sel, alt)}; the difference is {'statistically significant' if p < 0.05 else 'not statistically significant'} "
                f"(McNemar exact test, p = {p:.3f}). {sel} has {compare(sel, alt, 'Params (M)', 'parameters', 'fewer', 'more')}, "
                f"needs {compare(sel, alt, 'GFLOPs', 'FLOPs', 'fewer', 'more')} and runs {speed(sel, alt)} on one CPU thread.")

rep = report.loc[CLASSES]
worst, strongest = rep['recall'].idxmin(), rep['recall'].idxmax()
sentence = (f"**Per class.** Recall ranged from {rep.loc[worst, 'recall']:.2f} for {worst} to "
            f"{rep.loc[strongest, 'recall']:.2f} for {strongest}. ")
if worst == 'trash':
    sentence += (f"Trash is the smallest class ({int(n_train[TRASH])} training images) and the most varied, "
                 f"since it covers everything non-recyclable. ")
sentence += (f"The most frequent confusion was {top_confusions[0][1]} predicted as {top_confusions[0][2]} "
             f"({top_confusions[0][0]} images)." if top_confusions else "No test image was misclassified.")
text.append(sentence)

text.append(f"**Sorting impact.** {pct(R[sel]['Contamination'])} of the items routed to a recycling stream were residual trash, "
            f"{pct(R[sel]['Recyclable loss'])} of recyclable items would have gone to residual waste, and "
            f"{pct(R[sel]['Cross-material error'])} of recyclables went to the wrong material stream.")
if pd.notna(op['test accuracy']):
    text.append(f"**Reject option.** With a confidence threshold of {THR:.2f}, chosen on the validation set, the model sorts "
                f"{pct(op['test coverage'])} of test items automatically at {pct(op['test accuracy'])} accuracy and sends the "
                f"rest to manual sorting.")
if DEPLOY is not None:
    q, f = DEPLOY.loc['ONNX INT8'], DEPLOY.loc['PyTorch FP32']
    text.append(f"**Deployment optimisation.** INT8 quantisation changed the model file from {f['Size (MB)']:.1f} MB to "
                f"{q['Size (MB)']:.1f} MB and single-thread CPU latency from {f['CPU latency (ms)']:.1f} ms (PyTorch FP32) to "
                f"{q['CPU latency (ms)']:.1f} ms. Test accuracy was {pct(q['Accuracy'])} against {pct(f['Accuracy'])} in FP32, "
                f"and {pct(q['Agreement with PyTorch'])} of predictions were unchanged.")

conclusion = (f"**Conclusion.** {sel} was chosen by a rule fixed before testing: the highest validation macro-F1 within "
              f"the {CFG['gflops_budget']:g}-GFLOP edge budget. ")
if sel != 'ResNet-50' and 'ResNet-50' in MCN:
    heavier = compare(sel, 'ResNet-50', 'GFLOPs', 'FLOPs', 'fewer', 'more')
    if MCN['ResNet-50'] >= 0.05:
        conclusion += (f"Its accuracy is statistically indistinguishable from ResNet-50 while needing {heavier}, "
                       f"so it delivers reference-level accuracy on hardware that fits inside a smart bin or beside an MRF conveyor.")
    elif R[sel]['Accuracy'] < R['ResNet-50']['Accuracy']:
        conclusion += (f"ResNet-50 is significantly more accurate but needs {compare('ResNet-50', sel, 'GFLOPs', 'FLOPs', 'fewer', 'more')} "
                       f"and exceeds the edge budget. ResNet-50 suits an MRF with a server or GPU; {sel} suits bins and "
                       f"embedded sorters.")
    else:
        conclusion += f"It is also significantly more accurate than ResNet-50 while needing {heavier}."
text.append(conclusion)

summary_md = '\n\n'.join(text)
report_cols = ['Accuracy', 'Precision (macro)', 'Recall (macro)', 'F1 (macro)', 'ROC-AUC (macro OvR)', 'mAP (macro AP)',
               'Params (M)', 'GFLOPs', 'CPU latency (ms)']
try:
    table_md = table[report_cols].round(3).to_markdown()
except ImportError:
    table_md = table[report_cols].round(3).to_string()
(OUT_DIR / 'results_summary.md').write_text('## Results\n\n' + summary_md + '\n\n' + table_md + '\n', encoding='utf-8')
display(Markdown(summary_md))
print(table_md)

In [ ]:
# 27. Package the outputs
bundle = Path('smart_waste_results')
shutil.rmtree(bundle, ignore_errors=True)
bundle.mkdir()
for f in list(OUT_DIR.glob('*.csv')) + list(OUT_DIR.glob('*.md')):
    shutil.copy(f, bundle / f.name)
shutil.copytree(FIG_DIR, bundle / 'figures')
for f in (MODEL_DIR / f'{slug(SELECTED)}.pt', MODEL_DIR / 'selected_int8.onnx'):
    if f.exists():
        shutil.copy(f, bundle / f.name)
archive = shutil.make_archive(str(bundle), 'zip', bundle)
print(f'{archive}: {Path(archive).stat().st_size / 1e6:.1f} MB')
try:
    from google.colab import files
    files.download(archive)
except ImportError:
    pass